<a href="https://colab.research.google.com/github/frank-morales2020/MLxDL/blob/main/h2e_vjepa2_claude4dot7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q timm einops
!pip install -q git+https://github.com/facebookresearch/vjepa2.git
!pip install -q anthropic
!pip install colab-env -q
!pip install -q av torch numpy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
!nvidia-smi

Thu Apr 16 18:50:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   28C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
# =============================================================================
# PRE-REQUISITES — run these cells in Colab BEFORE this script
# =============================================================================
# !pip install -q timm einops
# !pip install -q git+https://github.com/facebookresearch/vjepa2.git
# !pip install -q anthropic
# =============================================================================

import torch
import torch.nn as nn
import torch.optim as optim
import os
import io
import base64
import numpy as np
import random
import av
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from google.colab import userdata
import anthropic
from huggingface_hub import login
from transformers import AutoVideoProcessor, AutoModel
import warnings

warnings.filterwarnings("ignore")


# =============================================================================
# 0. HUGGING FACE AUTHENTICATION
# =============================================================================
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN, add_to_git_credential=False)
print("🤗 Hugging Face authentication successful.")


# =============================================================================
# 1. H2E DETERMINISM (Seed 123)
# =============================================================================
def set_reproducibility(seed=123):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"🔐 H2E Determinism Locked | Seed: {seed}")

set_reproducibility(123)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device: {DEVICE}")


# =============================================================================
# 2. CONFIGURATION
# =============================================================================
VIDEO_PATH = (
    "/content/drive/MyDrive/datasets/TartanAviation/vision/"
    "1_2023-02-22-15-21-49/1_2023-02-22-15-21-49.mp4"
)

# ── Official HuggingFace repos ────────────────────────────────────────────────
# Source: https://github.com/facebookresearch/vjepa2?tab=readme-ov-file
# facebook/vjepa2-vitl-fpc64-256   ← ViT-L, 300M params  [USED]
# facebook/vjepa2-vith-fpc64-256   ← ViT-H, 600M params
# facebook/vjepa2-vitg-fpc64-256   ← ViT-g, 1B   params
# facebook/vjepa2-vitg-fpc64-384   ← ViT-g, 1B   params @ 384px (best quality)
HF_REPO = "facebook/vjepa2-vitl-fpc64-256"

# ── Claude model selection ────────────────────────────────────────────────────
# claude-opus-4-6           → most capable, best for safety-critical reasoning
# claude-sonnet-4-6         → fast + smart, best balance for production
# claude-haiku-4-5-20251001 → fastest, lightest

#CLAUDE_MODEL         = "claude-opus-4-6"   # matches MODEL_NAME in your setup cell

CLAUDE_MODEL         = "claude-opus-4-7"
NUM_KEYFRAMES_CLAUDE = 4      # keyframes forwarded to Claude for visual grounding

SROI_THRESHOLD       = 0.75   # realistic for all-MiniLM-L6-v2 cosine space
NUM_FRAMES           = 16     # V-JEPA 2 default temporal sampling window
NUM_KEYFRAMES_GEMINI = 4      # alias kept for extract_video_data compatibility
VJEPA_EMBED_DIM      = 1024   # ViT-L hidden dim (ViT-H=1280, ViT-g=1408)


# =============================================================================
# 3. V-JEPA 2 MODEL LOADER
# =============================================================================
def load_vjepa2(hf_repo: str = HF_REPO):
    """
    Load V-JEPA 2 via HuggingFace transformers.

    torch.hub is intentionally skipped — Colab's egress proxy blocks
    outbound traffic to dl.fbaipublicfiles.com (redirects to localhost:8300).
    HuggingFace is the reliable path in this environment.

    Official repos: https://github.com/facebookresearch/vjepa2
    Input  : pixel_values_videos [B, T, C, H, W]
    Output : last_hidden_state   [B, N, D]  → mean-pooled → [B, D]
    """
    print(f"📥 Loading V-JEPA 2 | {hf_repo} ...")
    try:
        processor = AutoVideoProcessor.from_pretrained(
            hf_repo,
            token=HF_TOKEN,
        )
        model = AutoModel.from_pretrained(
            hf_repo,
            token               = HF_TOKEN,
            dtype               = torch.float16,
            device_map          = "auto",
            attn_implementation = "sdpa",
        )
        model.eval()
        print(f"✅ V-JEPA 2 loaded | dtype: {next(model.parameters()).dtype}")
        return model, processor
    except Exception as exc:
        print(f"⚠️  HuggingFace load failed: {exc}")
        print("   → Activating stub model for pipeline testing.")
        return None, None


def build_stub_vjepa2(embed_dim: int = VJEPA_EMBED_DIM):
    """
    Stand-in that mimics V-JEPA 2 output shape.
    Returns (model, processor) — processor is None for the stub.
    """
    class _StubVJEPA(nn.Module):
        def forward(self, pixel_values_videos=None, **kw):
            B = pixel_values_videos.shape[0] if pixel_values_videos is not None else 1
            hidden = torch.randn(B, 1, embed_dim, device=DEVICE)
            return type("Out", (), {"last_hidden_state": hidden})()

    stub = _StubVJEPA().to(DEVICE).eval()
    print(f"🔧 Stub V-JEPA active (embed_dim={embed_dim}). "
          "Replace with real model in production.")
    return stub, None


# =============================================================================
# 4. H2E INDUSTRIAL CONTROLLER
# =============================================================================
class H2EController:
    """
    H2E perception-action loop.

    Pipeline
    ────────
    Video  ──► V-JEPA 2  ──► visual_embedding  [1, embed_dim]
                                     │
                               visual_projector          expert_text_vector
                                     │                          │
                             proj_embedding ──cosine──► visual_SROI
    Video keyframes ──► Gemini ──► action_text
                                     │
                         action_embedding ──cosine──► text_SROI
                                                           │
                                             fused_SROI = mean(visual, text)
                                                           │
                                            < threshold? → Nested Learning
    """

    def __init__(self, model, processor, expert_map: dict,
                 embed_dim: int = VJEPA_EMBED_DIM):

        self.model     = model
        self.processor = processor
        self.embed_dim = embed_dim

        # ── Text embedder ─────────────────────────────────────────────────────
        self.embedder = SentenceTransformer("all-MiniLM-L6-v2")
        text_dim      = self.embedder.get_sentence_embedding_dimension()  # 384

        self.expert_text_vectors = {
            k: self.embedder.encode(v) for k, v in expert_map.items()
        }

        # ── Visual → text projector (Nested Learning target) ──────────────────
        self.visual_projector = nn.Linear(embed_dim, text_dim).to(DEVICE)
        nn.init.xavier_uniform_(self.visual_projector.weight)
        nn.init.zeros_(self.visual_projector.bias)
        self.visual_projector.eval()

        self.slow_optimizer = optim.Adam(
            self.visual_projector.parameters(), lr=1e-3
        )

        # ── Claude client ──────────────────────────────────────────────────────
        #api_key = os.environ.get("ANTHROPIC_API_KEY")

        from google.colab import userdata
        api_key = userdata.get('ANTHROPIC_API_KEY')

        if not api_key:
            raise ValueError("ANTHROPIC_API_KEY is not set.")
        self.claude = anthropic.Anthropic(api_key=api_key)
        print(f"Claude client configured for **{CLAUDE_MODEL}**.")

    # ──────────────────────────────────────────────────────────────────────────
    # VIDEO PROCESSING
    # ──────────────────────────────────────────────────────────────────────────
    def extract_video_data(self, path: str) -> tuple:
        """
        Decode video and return:
          pixel_values : torch.Tensor [1, T, C, H, W]  ← V-JEPA 2 format
          keyframes    : list[PIL.Image]                ← for Gemini grounding
        """
        container = av.open(path)
        try:
            frames = [f.to_image() for f in container.decode(video=0)]
        finally:
            container.close()

        if not frames:
            raise ValueError(f"No frames decoded from: {path}")

        indices = np.linspace(0, len(frames) - 1, NUM_FRAMES, dtype=int)
        sampled = [frames[i] for i in indices]

        if self.processor is not None:
            # Official processor handles resize + normalise
            # AutoVideoProcessor accepts [T, C, H, W] uint8 numpy array
            video_np     = np.stack([np.array(f) for f in sampled])   # [T,H,W,C]
            video_np     = video_np.transpose(0, 3, 1, 2)             # [T,C,H,W]
            inputs       = self.processor(list(video_np), return_tensors="pt")
            pixel_values = inputs["pixel_values_videos"].to(DEVICE)   # [1,T,C,H,W]
        else:
            # Stub fallback: manual resize + normalise
            frame_tensors = torch.stack([
                torch.from_numpy(np.array(f)).permute(2, 0, 1).float() / 255.0
                for f in sampled
            ])  # [T, C, H, W]
            frame_tensors = torch.nn.functional.interpolate(
                frame_tensors, size=(256, 256), mode="bilinear", align_corners=False
            )
            pixel_values = frame_tensors.unsqueeze(0).to(DEVICE)      # [1,T,C,H,W]

        kf_idx    = np.linspace(0, len(sampled) - 1, NUM_KEYFRAMES_GEMINI, dtype=int)
        keyframes = [sampled[i] for i in kf_idx]

        return pixel_values, keyframes

    # ──────────────────────────────────────────────────────────────────────────
    # VISUAL FEATURE EXTRACTION
    # ──────────────────────────────────────────────────────────────────────────
    def extract_visual_embedding(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """
        V-JEPA 2 forward pass.
        last_hidden_state: [B, N, D] → mean-pool → [B, D] → L2-norm → [B, D]
        Cast fp16 → fp32 before projector.
        """
        with torch.no_grad():
            outputs = self.model(pixel_values_videos=pixel_values)
        emb = outputs.last_hidden_state          # [B, N, D]
        if emb.dim() == 3:
            emb = emb.mean(dim=1)               # [B, D]
        emb = emb.to(torch.float32)
        return torch.nn.functional.normalize(emb, dim=-1)

    # ──────────────────────────────────────────────────────────────────────────
    # CLAUDE CALL
    # ──────────────────────────────────────────────────────────────────────────
    def call_claude(self, keyframes: list, prompt: str) -> str:
        """
        Call Claude with keyframe images + text prompt.

        Claude's messages API accepts images as base64-encoded content blocks
        directly in the user message — no separate upload step needed.

        Supports vision natively across all claude-opus / sonnet / haiku models.
        """
        try:
            # Build content blocks: [image, image, ..., text]
            content = []
            for frame in keyframes:
                buf = io.BytesIO()
                frame.save(buf, format="JPEG", quality=85)
                img_b64 = base64.standard_b64encode(buf.getvalue()).decode("utf-8")
                content.append({
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/jpeg",
                        "data": img_b64,
                    }
                })
            content.append({"type": "text", "text": prompt})

            response = self.claude.messages.create(
                model      = CLAUDE_MODEL,
                max_tokens = 1024,
                system     = (
                    "You are an expert aviation safety controller. "
                    "Provide precise, protocol-compliant safety decisions."
                ),
                messages   = [{"role": "user", "content": content}],
            )
            return response.content[0].text.strip()

        except anthropic.APIError as e:
            print(f"⚠️  Claude APIError ({e}). Returning fallback action.")
            return "ACTION: Maintain safe hold. EXPLANATION: Claude API unavailable."
        except Exception as e:
            print(f"⚠️  Claude unexpected error ({e}). Returning fallback action.")
            return "ACTION: Maintain safe hold. EXPLANATION: General API error."

    # ──────────────────────────────────────────────────────────────────────────
    # FUSED SROI
    # ──────────────────────────────────────────────────────────────────────────
    def compute_sroi(
        self,
        visual_embedding: torch.Tensor,
        action_text: str,
        scenario_key: str,
    ) -> tuple[float, float, float]:
        """
        Returns (fused_sroi, visual_sroi, text_sroi).

        visual_sroi = cosine(projected_visual, expert_text_vec)
        text_sroi   = cosine(action_text_emb,  expert_text_vec)
        fused_sroi  = 0.5 * visual_sroi + 0.5 * text_sroi
        """
        expert_vec = self.expert_text_vectors[scenario_key]

        with torch.no_grad():
            proj    = self.visual_projector(visual_embedding)
        visual_sroi = float(cosine_similarity(proj.cpu().numpy(), [expert_vec])[0][0])

        action_vec  = self.embedder.encode(action_text)
        text_sroi   = float(cosine_similarity([action_vec], [expert_vec])[0][0])

        fused_sroi  = 0.5 * visual_sroi + 0.5 * text_sroi
        return fused_sroi, visual_sroi, text_sroi

    # ──────────────────────────────────────────────────────────────────────────
    # NESTED LEARNING
    # ──────────────────────────────────────────────────────────────────────────
    def nested_learning_step(
        self,
        visual_embedding: torch.Tensor,
        scenario_key: str,
        n_steps: int = 10,
    ) -> None:
        """
        Adapts visual_projector to close the SROI gap.
        V-JEPA 2 backbone stays frozen — only the projector is updated.
        Loss: cosine distance between projected embedding and expert vector.
        """
        expert_vec = torch.tensor(
            self.expert_text_vectors[scenario_key],
            dtype=torch.float32, device=DEVICE,
        ).unsqueeze(0)

        self.visual_projector.train()
        for step in range(1, n_steps + 1):
            self.slow_optimizer.zero_grad()
            proj     = self.visual_projector(visual_embedding.detach())
            proj_n   = torch.nn.functional.normalize(proj, dim=-1)
            expert_n = torch.nn.functional.normalize(expert_vec, dim=-1)
            loss     = 1.0 - (proj_n * expert_n).sum()   # cosine dist ∈ [0, 2]
            loss.backward()
            self.slow_optimizer.step()
            if step % 10 == 0 or step == n_steps:
                print(f"      [Nested Learning] step {step:>2}/{n_steps} "
                      f"| loss: {loss.item():.4f}")
        self.visual_projector.eval()

    # ──────────────────────────────────────────────────────────────────────────
    # MAIN CYCLE
    # ──────────────────────────────────────────────────────────────────────────
    def run_cycle(self, video_path: str, goal: str, scenario_key: str) -> str:
        print(f"\n{'='*65}")
        print(f"🚀 H2E SYSTEM CYCLE")
        print(f"   Scenario : {scenario_key}")
        print(f"   Goal     : {goal}")
        print(f"   Claude   : {CLAUDE_MODEL}")
        print(f"{'='*65}")

        # ── Phase 1 : Video decoding + V-JEPA 2 ──────────────────────────────
        pixel_values, keyframes = self.extract_video_data(video_path)
        print(f"\n[Phase 1] pixel_values shape : {tuple(pixel_values.shape)}"
              f"  (B, T, C, H, W) ✅")

        visual_embedding = self.extract_visual_embedding(pixel_values)
        print(f"[Phase 1] Visual embedding   : {tuple(visual_embedding.shape)}")

        # ── Phase 2 : Claude multimodal reasoning ─────────────────────────────
        prompt = (
            f"You are an expert aviation safety controller.\n"
            f"Scenario: {scenario_key}\n"
            f"Goal: {goal}\n\n"
            f"Examine the {NUM_KEYFRAMES_GEMINI} attached video keyframes "
            f"and respond with:\n"
            f"ACTION: <concise recommended action>\n"
            f"EXPLANATION: <safety rationale, max 3 sentences>"
        )
        print(f"[Phase 2] {NUM_KEYFRAMES_GEMINI} keyframes ready for Claude ✅")
        action_text = self.call_claude(keyframes, prompt)
        print(f"\n[Phase 2] Claude response received ✅")

        # ── Phase 3 : Fused SROI accountability ──────────────────────────────
        fused, v_sroi, t_sroi = self.compute_sroi(
            visual_embedding, action_text, scenario_key
        )
        print(f"\n[Phase 3] SROI Accountability Report")
        print(f"   Visual SROI : {v_sroi:.4f}")
        print(f"   Text SROI   : {t_sroi:.4f}")
        print(f"   Fused SROI  : {fused:.4f}  (threshold: {SROI_THRESHOLD})")

        # ── Phase 4 : Nested Learning if SROI insufficient ────────────────────
        if fused < SROI_THRESHOLD:
            print(f"\n[Phase 4] 🛠️  Fused SROI below threshold — "
                  f"triggering Nested Learning (SLOW adaptation)...")
            self.nested_learning_step(visual_embedding, scenario_key, n_steps=100)
            fused, v_sroi, t_sroi = self.compute_sroi(
                visual_embedding, action_text, scenario_key
            )
            print(f"   [Post-Adaptation] Fused SROI: {fused:.4f}")
        else:
            print(f"\n[Phase 4] ✅ SROI above threshold — no adaptation required.")

        return action_text

    # ──────────────────────────────────────────────────────────────────────────
    # ──────────────────────────────────────────────────────────────────────────
    # HELPERS
    # ──────────────────────────────────────────────────────────────────────────
    # Note: image encoding is handled inside call_claude() directly.
    # Claude's API accepts base64 content blocks in the messages array —
    # no separate part-building step is needed.


# =============================================================================
# 5. EXPERT INTENT LIBRARY
# =============================================================================
EXPERT_INTENTS = {
    "emergency landing": (
        "As an expert controller I verify gear status, declare emergency, "
        "clear all airspace, coordinate ARFF standby, and confirm runway availability."
    ),
    "airplane landing": (
        "Verify runway clearance, confirm gear down and locked, "
        "maintain glideslope stability, and issue landing clearance."
    ),
}

🤗 Hugging Face authentication successful.
🔐 H2E Determinism Locked | Seed: 123
🖥️  Device: cuda


https://platform.claude.com/docs/en/about-claude/models/overview

In [ ]:
# =============================================================================
# 6. EXECUTION
# =============================================================================
vjepa_model, vjepa_processor = load_vjepa2(HF_REPO)

if vjepa_model is None:
    vjepa_model, vjepa_processor = build_stub_vjepa2(VJEPA_EMBED_DIM)

h2e = H2EController(
    model      = vjepa_model,
    processor  = vjepa_processor,
    expert_map = EXPERT_INTENTS,
    embed_dim  = VJEPA_EMBED_DIM,
)

In [5]:
final_action = h2e.run_cycle(
    video_path   = VIDEO_PATH,
    goal         = "Resolve landing gear unsafe indication and execute emergency landing.",
    scenario_key = "emergency landing",
)

print(f"\n{'='*65}")
print(f"[Final Agent Action]:\n{final_action}")
print(f"{'='*65}")


🚀 H2E SYSTEM CYCLE
   Scenario : emergency landing
   Goal     : Resolve landing gear unsafe indication and execute emergency landing.
   Claude   : claude-opus-4-7

[Phase 1] pixel_values shape : (1, 16, 3, 256, 256)  (B, T, C, H, W) ✅
[Phase 1] Visual embedding   : (1, 1024)
[Phase 2] 4 keyframes ready for Claude ✅

[Phase 2] Claude response received ✅

[Phase 3] SROI Accountability Report
   Visual SROI : 0.0362
   Text SROI   : 0.5802
   Fused SROI  : 0.3082  (threshold: 0.75)

[Phase 4] 🛠️  Fused SROI below threshold — triggering Nested Learning (SLOW adaptation)...
      [Nested Learning] step 10/100 | loss: 0.0420
      [Nested Learning] step 20/100 | loss: 0.0122
      [Nested Learning] step 30/100 | loss: 0.0033
      [Nested Learning] step 40/100 | loss: 0.0011
      [Nested Learning] step 50/100 | loss: 0.0004
      [Nested Learning] step 60/100 | loss: 0.0002
      [Nested Learning] step 70/100 | loss: 0.0001
      [Nested Learning] step 80/100 | loss: 0.0000
      [Nested